In [1]:
#rag_pipeline() Using TinyLLaMA

In [7]:
import pandas as pd
import faiss
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import numpy as np

# Load chunk texts
df = pd.read_csv(r"C:\Users\hp\Downloads\filtered_chunks.csv")
texts = df["chunk_text"].tolist()

# Load FAISS index
index = faiss.read_index(r"C:/Users/hp/Downloads/faiss_index1a.idx")

# Load embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Load TinyLLaMA model + tokenizer
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
llama_model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
).to("cuda" if torch.cuda.is_available() else "cpu")


In [9]:
def retrieve_top_k_chunks(query, k=5):
    query_vector = embed_model.encode([query]).astype("float32")
    D, I = index.search(query_vector, k)
    return [texts[i] for i in I[0]]


In [10]:
def build_prompt(query, chunks):
    mapped = "\n\n---\n\n".join([f"{chunk}\n\nQuestion: {query}" for chunk in chunks])
    final_prompt = (
        "Answer the question below using the provided context. "
        "Think step by step and explain your reasoning.\n\n" + mapped
    )
    return final_prompt


In [11]:
def call_tiny_llama(prompt):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(llama_model.device)
    outputs = llama_model.generate(**inputs, max_new_tokens=256)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [12]:
def rag_pipeline(query):
    chunks = retrieve_top_k_chunks(query)
    prompt = build_prompt(query, chunks)
    answer = call_tiny_llama(prompt)
    return answer


In [14]:
query = "What are the symptoms of heat stroke?"
answer = rag_pipeline(query)

print("\n Final Answer:")
print(answer)



 Final Answer:
Answer the question below using the provided context. Think step by step and explain your reasoning.

o connor putting the heat to the night. weinstein, steve february 15, 1989. in the heat of the night sends a message popular nbc series gives positive role model of race relations, says producer. park, jeannie armstrong, lois may 7, 1990. in the heat of the night s eerie parallels to her sister s murder allow actress denise nicholas to finally conquer her grief. 846 q connor to sue tabloid for rollins story. tv guide, july 14 20, 2001 7. in the heat of the night sweet, sweet blues season 5, episode 8. usc news april 5, 2016. the civil rights experience of novelist denise nicholas inspired her artistry 9. kloer, phil may 6, 1993. howard rollins in seclusion, his acting career in jeopardy. in the heat of the night complete season 8 the final season. retrieved march 18, 2015. carroll o connor s son kills himself at 33. in the heat of the night the first season. in the heat

In [ ]:
#Caching & Fallback

In [16]:
!pip install diskcache


Defaulting to user installation because normal site-packages is not writeable
  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
Using cached diskcache-5.6.3-py3-none-any.whl (45 kB)


In [17]:
from diskcache import Cache

cache = Cache("./llm_cache")

def cached_response(query):
    if query in cache:
        return cache[query]
    else:
        chunks = retrieve_top_k_chunks(query)
        prompt = build_prompt(query, chunks)
        answer = call_llm(prompt)
        cache[query] = answer
        return answer


In [ ]:
#LLM Fallback Strategy

In [19]:
def call_llm(prompt):
    try:
        return call_openai(prompt)
    except Exception as e:
        print(" OpenAI failed. Falling back to TinyLLaMA.")
        return call_tiny_llama(prompt)


In [20]:
from diskcache import Cache

# Create or load a cache folder
cache = Cache("./llm_cache")

# Example usage
query = "What are the symptoms of heat stroke?"
if query in cache:
    answer = cache[query]
else:
    answer = rag_pipeline(query)
    cache[query] = answer

print(answer)


Answer the question below using the provided context. Think step by step and explain your reasoning.

o connor putting the heat to the night. weinstein, steve february 15, 1989. in the heat of the night sends a message popular nbc series gives positive role model of race relations, says producer. park, jeannie armstrong, lois may 7, 1990. in the heat of the night s eerie parallels to her sister s murder allow actress denise nicholas to finally conquer her grief. 846 q connor to sue tabloid for rollins story. tv guide, july 14 20, 2001 7. in the heat of the night sweet, sweet blues season 5, episode 8. usc news april 5, 2016. the civil rights experience of novelist denise nicholas inspired her artistry 9. kloer, phil may 6, 1993. howard rollins in seclusion, his acting career in jeopardy. in the heat of the night complete season 8 the final season. retrieved march 18, 2015. carroll o connor s son kills himself at 33. in the heat of the night the first season. in the heat of the night th